`requirements/gpu.txt`

In [6]:
import pandas as pd, gc, rustworkx as rx
from qiskit_aer.noise import NoiseModel
from qiskit_aer import AerSimulator
from qiskit_experiments.library import QuantumVolume
from collections import defaultdict
from calibration_helpers import load_payload

In [7]:
def noisy_simulator_from_payload(payload):
    '''Creates a noisy AerSimulator from the noise model and coupling map in the payload.'''
    
    noise_model = NoiseModel.from_dict(payload["noise_model"])
    coupling_map = payload.get("coupling_map")
    return AerSimulator(
        noise_model=noise_model,
        coupling_map=coupling_map,
        basis_gates=noise_model.basis_gates,
        method="automatic",
        device="GPU",
    )


def run_qv_for_subset(backend, ideal_backend, subset, shots=2000, trials=100, seed=42):
    '''Runs the Quantum Volume experiment for a given subset of qubits and returns the results.'''

    exp = QuantumVolume(
        physical_qubits=list(subset),
        trials=trials,
        seed=seed,
        simulation_backend=ideal_backend,
    )

    exp.set_run_options(shots=shots)

    exp.set_transpile_options(
        coupling_map=backend.coupling_map,
        layout_method="trivial",
        routing_method="basic",
        optimization_level=0,
        seed_transpiler=seed,
        num_processes=0,
    )

    return exp.run(backend).block_for_results()

In [8]:
def build_undirected_adj(coupling_map) -> dict[int, set[int]]:
    """
    Returns adjacency dict: {q: set(neighbors)}
    """
    adj = defaultdict(set)
    for a, b in coupling_map:
        a = int(a); b = int(b)
        adj[a].add(b)
        adj[b].add(a)
    return dict(adj)


def connected_n_tuples_from_coupling(coupling_map, n: int):
    """
    Returns list of n-tuples (sorted tuples) that are connected (induced subgraph is connected).
    """

    nodes = sorted({int(a) for a, _ in coupling_map} | {int(b) for _, b in coupling_map})
    if n == 1:
        return [(q,) for q in nodes]
    if len(nodes) < n:
        return []

    # build mapping: qubit label -> rustworkx node index
    g = rx.PyGraph(multigraph=False)
    label_to_idx = {}
    for q in nodes:
        label_to_idx[q] = g.add_node(q)  # store the label as node payload

    # add edges
    for a, b in coupling_map:
        a = int(a); b = int(b)
        g.add_edge(label_to_idx[a], label_to_idx[b], None)

    # enumerate connected k-node subgraphs (returns lists of node indices)
    subgraphs = rx.connected_subgraphs(g, n)

    # convert node indices back to qubit labels (payloads), sort each tuple
    out = []
    for idxs in subgraphs:
        labels = sorted(g[idx] for idx in idxs)
        out.append(tuple(labels))

    # ensure uniqueness + deterministic ordering
    return sorted(set(out))

In [ ]:
def run_qv_over_connected_n_tuples(calibration_path, n: int, shots=2000, trials=100, seed=42):
    '''Runs Quantum Volume experiments over all connected n-tuples of qubits and returns a summary of results.'''

    payload = load_payload(calibration_path)
    coupling_map = payload.get("coupling_map")

    tuples_ = connected_n_tuples_from_coupling(coupling_map, n)

    backend = noisy_simulator_from_payload(payload)
    backend.set_options(batched_shots_gpu=True)
    ideal_backend = AerSimulator(method="statevector", device="GPU")

    summaries = []
    for i, subset in enumerate(tuples_, 1):
        print(f"[{i}/{len(tuples_)}] running QV on subset {subset}")

        exp_data = run_qv_for_subset(
            backend=backend,
            ideal_backend=ideal_backend,
            subset=subset,
            shots=shots,
            trials=trials,
            seed=seed,
        )

        df = exp_data.analysis_results(dataframe=True)
        hop_row = df[df["name"] == "mean_HOP"].iloc[0]

        mean_hop = hop_row["value"].nominal_value
        hop_err  = hop_row["value"].std_dev

        summaries.append((subset, mean_hop, hop_err))

        del exp_data, df
        gc.collect()

    return summaries

In [ ]:
cal_path = "calibrations/ibm_marrakesh/20260129_101824.json"

qubits = 5

expdata = run_qv_over_connected_n_tuples(
    cal_path,
    n=qubits,
    shots=100,
    trials=100,
    seed=42,
)

df = pd.DataFrame(expdata, columns=["subset", "mean_HOP", "hop_error"])
df = df.sort_values("mean_HOP", ascending=False).reset_index(drop=True)

out_csv = f"results/ibm_marrakesh_qv_{qubits}_tuples.csv"
df.to_csv(out_csv, index=False)

/tmp/ipykernel_1514/2068522216.py:4: DeprecationWarning: from_dict has been deprecated as of qiskit-aer 0.15.0 and will be removed no earlier than 3 months from that release date.
  noise_model = NoiseModel.from_dict(payload["noise_model"])


[1/566] running QV on subset (0, 1, 2, 3, 4)
[2/566] running QV on subset (0, 1, 2, 3, 16)
[3/566] running QV on subset (1, 2, 3, 4, 5)
[4/566] running QV on subset (1, 2, 3, 4, 16)
[5/566] running QV on subset (1, 2, 3, 16, 23)
[6/566] running QV on subset (2, 3, 4, 5, 6)
